# REFUGE2 Optic Disc/Cup — Boundary Loss Ablation

3-class 세그멘테이션(BG / Disc / Cup)에서 Boundary Loss 효과 검증.

비교 실험 (5종):
| # | loss_name | 설명 |
|---|---|---|
| 1 | `ce_dice` | 기존 베이스라인 |
| 2 | `plwce_dice` | 기존 PLWCE 베스트 |
| 3 | `ce_dice_boundary` | 문헌 베이스라인 (Kervadec 2019) |
| 4 | `plwce_dice_boundary` | **핵심 실험** |
| 5 | `plwce_boundary` | Dice 제거 실험 (성능 하락 예상) |

> 결과 저장: `medical_data/boundary_ablation/results/refuge/`

In [1]:
# === Cell 0: 환경설정 ===
import subprocess, sys
for pkg in ['segmentation-models-pytorch', 'openpyxl', 'scipy', 'optuna']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, warnings, json, random, shutil, zipfile
warnings.filterwarnings('ignore')
os.environ['TQDM_DISABLE'] = '1'

import numpy as np
import cv2
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import optuna

_CL_LOCAL = '/root/imbalanced-data-LWCE/medical_data'
_CL_COLAB = '/tmp/custom_losses'
sys.path.insert(0, _CL_LOCAL if os.path.exists(_CL_LOCAL) else _CL_COLAB)
from custom_losses import get_loss_function

GDRIVE_ZIP = '/content/drive/MyDrive/imbalanced-data-LWCE/refuge/refuge.zip'
IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    _src = '/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/custom_losses.py'
    if not os.path.exists(os.path.join(_CL_LOCAL, 'custom_losses.py')) and os.path.exists(_src):
        os.makedirs(_CL_COLAB, exist_ok=True)
        shutil.copy(_src, _CL_COLAB)
    print('Google Drive 마운트 완료')
except Exception:
    print('Colab 환경 아님')

DOMAIN      = 'refuge'
NUM_CLASSES = 3
CLASS_NAMES = ['BG', 'Disc', 'Cup']
IMG_SIZE    = 512
BATCH_SIZE  = 8
NUM_WORKERS = 0
SEED        = 42

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/refuge'
os.makedirs(RESULTS_DIR, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경설정 완료')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive 마운트 완료
Device: cuda
환경설정 완료


In [2]:
# === Cell 1: 데이터 로드 ===

DATA_DIR     = '/tmp/refuge_data'
CHECK_SUBDIR = 'Training400'

if not os.path.exists(os.path.join(DATA_DIR, CHECK_SUBDIR)):
    if IS_COLAB and os.path.exists(GDRIVE_ZIP):
        print(f'Drive에서 압축 해제 중: {GDRIVE_ZIP}')
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(GDRIVE_ZIP, 'r') as zf:
            zf.extractall(DATA_DIR)
        for _inner_name in ['REFUGE', 'refuge']:
            _inner = os.path.join(DATA_DIR, _inner_name)
            if os.path.isdir(_inner):
                for item in os.listdir(_inner):
                    shutil.move(os.path.join(_inner, item), DATA_DIR)
                os.rmdir(_inner)
                break
        print('압축 해제 완료')
    else:
        print('[데이터 없음] Drive에 refuge.zip 업로드 필요')

TRAIN_IMG_BASE  = os.path.join(DATA_DIR, 'Training400')
TRAIN_MASK_BASE = os.path.join(DATA_DIR, 'Annotation-Training400', 'Disc_Cup_Masks')
VAL_IMG_DIR     = os.path.join(DATA_DIR, 'REFUGE-Validation400')
VAL_MASK_DIR    = os.path.join(DATA_DIR, 'REFUGE-Validation400-GT', 'Disc_Cup_Masks')
TEST_IMG_DIR    = os.path.join(DATA_DIR, 'Test400')
TEST_MASK_BASE  = os.path.join(DATA_DIR, 'REFUGE-Test-GT', 'Disc_Cup_Masks')

def collect_training_pairs(img_base, mask_base):
    pairs = []
    for subfolder in ['Glaucoma', 'Non-Glaucoma']:
        img_dir  = os.path.join(img_base,  subfolder)
        mask_dir = os.path.join(mask_base, subfolder)
        if not os.path.isdir(img_dir):
            continue
        for fname in sorted(os.listdir(img_dir)):
            stem      = os.path.splitext(fname)[0]
            img_path  = os.path.join(img_dir, fname)
            mask_path = os.path.join(mask_dir, stem + '.bmp')
            if os.path.exists(mask_path):
                pairs.append((img_path, mask_path))
    return pairs

def collect_flat_pairs(img_dir, mask_dir):
    mask_map = {os.path.splitext(f)[0]: os.path.join(mask_dir, f)
                for f in os.listdir(mask_dir) if f.lower().endswith('.bmp')}
    pairs = []
    for fname in sorted(os.listdir(img_dir)):
        stem = os.path.splitext(fname)[0]
        if stem in mask_map:
            pairs.append((os.path.join(img_dir, fname), mask_map[stem]))
    return pairs

def collect_test_pairs(img_dir, mask_base):
    mask_lookup = {}
    for sub in ['G', 'N']:
        sub_dir = os.path.join(mask_base, sub)
        if os.path.isdir(sub_dir):
            for f in os.listdir(sub_dir):
                stem = os.path.splitext(f)[0]
                mask_lookup[stem] = os.path.join(sub_dir, f)
    pairs = []
    for fname in sorted(os.listdir(img_dir)):
        stem = os.path.splitext(fname)[0]
        if stem in mask_lookup:
            pairs.append((os.path.join(img_dir, fname), mask_lookup[stem]))
    return pairs

tr_pairs   = collect_training_pairs(TRAIN_IMG_BASE, TRAIN_MASK_BASE)
val_pairs  = collect_flat_pairs(VAL_IMG_DIR, VAL_MASK_DIR)
test_pairs = collect_test_pairs(TEST_IMG_DIR, TEST_MASK_BASE)
print(f'Train: {len(tr_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}')

def decode_mask(mask_bgr):
    if len(mask_bgr.shape) == 3:
        gray = cv2.cvtColor(mask_bgr, cv2.COLOR_BGR2GRAY)
    else:
        gray = mask_bgr
    mask = np.zeros_like(gray, dtype=np.int64)
    mask[gray > 200]                  = 2  # Cup
    mask[(gray > 50) & (gray <= 200)] = 1  # Disc
    return mask

class REFUGEDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs   = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img      = cv2.imread(img_path,  cv2.IMREAD_COLOR)
        mask_raw = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        img      = cv2.resize(img,      (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        mask_raw = cv2.resize(mask_raw, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        mask = decode_mask(mask_raw)
        img  = (img - IMAGENET_MEAN) / IMAGENET_STD
        if self.augment:
            if random.random() > 0.5:
                img  = np.fliplr(img).copy();  mask = np.fliplr(mask).copy()
            if random.random() > 0.5:
                img  = np.flipud(img).copy();  mask = np.flipud(mask).copy()
            k = random.randint(0, 3)
            if k > 0:
                img  = np.rot90(img,  k).copy(); mask = np.rot90(mask, k).copy()
        return (torch.from_numpy(img.transpose(2, 0, 1)).float(),
                torch.from_numpy(mask).long())

train_ds = REFUGEDataset(tr_pairs,   augment=True)
val_ds   = REFUGEDataset(val_pairs,  augment=False)
test_ds  = REFUGEDataset(test_pairs, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print('DataLoader 완료')

Drive에서 압축 해제 중: /content/drive/MyDrive/imbalanced-data-LWCE/refuge/refuge.zip
압축 해제 완료
Train: 400, Val: 400, Test: 400
DataLoader 완료


In [3]:
# === Cell 2: 클래스 비율 계산 ===
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, mask_t in tqdm(train_loader, desc='클래스 비율 계산'):
    for c in range(NUM_CLASSES):
        class_counts[c] += int((mask_t == c).sum())

total = class_counts.sum()
print('\n클래스 비율:')
for name, count in zip(CLASS_NAMES, class_counts):
    print(f'  {name}: {count:,} ({count / total * 100:.2f}%)')
for c in range(1, NUM_CLASSES):
    print(f'  BG:{CLASS_NAMES[c]} = {class_counts[0] / class_counts[c]:.1f}:1')
class_counts = class_counts.tolist()


클래스 비율:
  BG: 469,458 (0.45%)
  Disc: 1,308,197 (1.25%)
  Cup: 103,079,945 (98.30%)
  BG:Disc = 0.4:1
  BG:Cup = 0.0:1


In [4]:
# === Cell 3: 모델 및 평가 함수 ===

def build_model():
    return smp.Unet(
        encoder_name='resnet34', encoder_weights='imagenet',
        in_channels=3, classes=NUM_CLASSES, activation=None,
    ).to(device)

def compute_val_mdice(model, loader):
    """학습 중 빠른 mDice 계산 (early stopping 기준)"""
    model.eval()
    dice_sum  = np.zeros(NUM_CLASSES - 1)
    n_batches = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs).argmax(dim=1)
            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                p = (preds == c).float()
                t = (masks == c).float()
                inter = (p * t).sum().item()
                union = (p.sum() + t.sum()).item()
                if union > 0:
                    dice_sum[c_idx] += 2. * inter / (union + 1e-8)
            n_batches += 1
    return float((dice_sum / max(n_batches, 1)).mean())

def compute_val_metrics(model, loader):
    """최종 평가: mDice + 클래스별 Dice"""
    model.eval()
    dice_sum  = np.zeros(NUM_CLASSES - 1)
    n_batches = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs).argmax(dim=1)
            for c_idx, c in enumerate(range(1, NUM_CLASSES)):
                p = (preds == c).float()
                t = (masks == c).float()
                inter = (p * t).sum().item()
                union = (p.sum() + t.sum()).item()
                if union > 0:
                    dice_sum[c_idx] += 2. * inter / (union + 1e-8)
            n_batches += 1
    dice_per_class = dice_sum / max(n_batches, 1)
    return {
        'mDice':     float(dice_per_class.mean()),
        'Dice_Disc': float(dice_per_class[0]),
        'Dice_Cup':  float(dice_per_class[1]),
    }

print('모델 + 평가 함수 준비 완료')

모델 + 평가 함수 준비 완료


In [5]:
# === Cell 4: 학습 함수 ===

def train_model(loss_name, alpha=1.0, gamma=2.0, epochs=50, lr=1e-4,
                subset_ratio=1.0, tag=''):
    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    if subset_ratio < 1.0:
        n      = max(1, int(len(train_ds) * subset_ratio))
        sub_ds = torch.utils.data.Subset(train_ds, random.sample(range(len(train_ds)), n))
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader

    best_mdice = 0.0
    best_state = None
    history    = {'loss': [], 'val_mdice': []}
    ckpt_path  = f'/tmp/ba_refuge_{tag}_{loss_name}.pth'

    for epoch in range(epochs):
        # Boundary Loss annealing: BL 비중 0 → 0.5 선형 증가
        if criterion.boundary_loss is not None:
            alpha_t = min(epoch / epochs, 0.5)
            criterion.set_boundary_alpha(alpha_t)

        model.train()
        epoch_loss = 0.0
        for imgs, masks in tqdm(loader,
                                desc=f'[{tag}] {loss_name} Ep{epoch+1:02d}/{epochs}',
                                leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), masks)  # 3-class: logits 직접 전달
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        val_mdice = compute_val_mdice(model, val_loader)
        history['loss'].append(epoch_loss / len(loader))
        history['val_mdice'].append(val_mdice)

        if val_mdice > best_mdice:
            best_mdice = val_mdice
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, ckpt_path)

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_mdice

print('train_model() 준비 완료')

train_model() 준비 완료


In [6]:
# === Cell 5: Optuna — PLWCE alpha 탐색 (Boundary Loss 환경) ===
import traceback
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'

ALPHA_LOW    = 2.5
ALPHA_HIGH   = 15.0
PROXY_EPOCHS = 8
PROXY_RATIO  = 0.15
N_TRIALS     = 20

def make_objective(loss_name):
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, mdice = train_model(loss_name, alpha=alpha,
                                      epochs=PROXY_EPOCHS, subset_ratio=PROXY_RATIO,
                                      tag=f'trial{trial.number}')
            return mdice
        except Exception:
            traceback.print_exc()
            return None
    return objective

# --- plwce_dice_boundary ---
print('Optuna: plwce_dice_boundary ...')
sampler_pdb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()})
study_pdb = optuna.create_study(direction='maximize', sampler=sampler_pdb)
study_pdb.optimize(make_objective('plwce_dice_boundary'), n_trials=N_TRIALS)
best_trials_pdb = [t for t in study_pdb.trials if t.value is not None]
best_alpha_with_dice = (
    best_trials_pdb[int(np.argmax([t.value for t in best_trials_pdb]))].params['alpha']
    if best_trials_pdb else 5.0
)
print(f'  best alpha (with Dice): {best_alpha_with_dice:.3f}')

# --- plwce_boundary (no Dice) ---
print('Optuna: plwce_boundary ...')
sampler_pb = optuna.samplers.GridSampler(
    {'alpha': np.linspace(ALPHA_LOW, ALPHA_HIGH, N_TRIALS).tolist()})
study_pb = optuna.create_study(direction='maximize', sampler=sampler_pb)
study_pb.optimize(make_objective('plwce_boundary'), n_trials=N_TRIALS)
best_trials_pb = [t for t in study_pb.trials if t.value is not None]
best_alpha_without_dice = (
    best_trials_pb[int(np.argmax([t.value for t in best_trials_pb]))].params['alpha']
    if best_trials_pb else 5.0
)
print(f'  best alpha (no Dice):   {best_alpha_without_dice:.3f}')

optuna_data = {
    'plwce_dice_boundary': {'best_alpha': best_alpha_with_dice},
    'plwce_boundary':      {'best_alpha': best_alpha_without_dice},
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json'), 'w') as f:
    json.dump(optuna_data, f, indent=2)
print('Optuna 결과 저장 완료')

Optuna: plwce_dice_boundary ...
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Ge

In [7]:
# === Cell 6: Boundary Ablation 전체 학습 ===

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

# Optuna 결과 로드 (Cell 5 미실행 시 JSON fallback)
try:
    _ = best_alpha_with_dice
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_optuna.json')) as f:
            d = json.load(f)
        best_alpha_with_dice    = d['plwce_dice_boundary']['best_alpha']
        best_alpha_without_dice = d['plwce_boundary']['best_alpha']
        print(f'Optuna 로드: with_dice={best_alpha_with_dice:.3f}, no_dice={best_alpha_without_dice:.3f}')
    except FileNotFoundError:
        best_alpha_with_dice    = 5.0
        best_alpha_without_dice = 5.0
        print('Optuna 미실행 → fallback alpha=5.0')

experiments = [
    ('ce_dice',             1.0,                     'CE+Dice                      [baseline]'),
    ('plwce_dice',          best_alpha_with_dice,     f'PLWCE+Dice                   (α={best_alpha_with_dice:.3f}) [baseline]'),
    ('ce_dice_boundary',    1.0,                     'CE+Dice+BL                   [literature]'),
    ('plwce_dice_boundary', best_alpha_with_dice,     f'PLWCE+Dice+BL                (α={best_alpha_with_dice:.3f})'),
    ('plwce_boundary',      best_alpha_without_dice,  f'PLWCE+BL     (no Dice)        (α={best_alpha_without_dice:.3f})'),
]

all_results = {}
for loss_name, alpha, label in experiments:
    model, history, best_mdice = train_model(
        loss_name=loss_name, alpha=alpha,
        epochs=FINAL_EPOCHS, lr=FINAL_LR, tag='ba')
    all_results[label] = {
        'model': model, 'history': history, 'best_mdice': best_mdice,
        'loss_name': loss_name, 'alpha': alpha,
    }

print('\n' + '='*65)
print('[Boundary Ablation 요약 — Val mDice]')
print(f"{'Loss':<55} {'Val mDice':>9}")
print('-'*65)
for label, v in all_results.items():
    print(f"{label:<55} {v['best_mdice']:>9.4f}")

[ce_dice] Components: CE + DiceLoss  (λ=0.500 each)
[plwce_dice] CE weights (plwce): Generated.
[plwce_dice] Components: CE + DiceLoss  (λ=0.500 each)
[ce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_dice_boundary] CE weights (plwce): Generated.
[plwce_dice_boundary] Components: CE + DiceLoss + BoundaryLoss  (λ=0.333 each)
[plwce_boundary] CE weights (plwce): Generated.
[plwce_boundary] Components: CE + BoundaryLoss  (λ=0.500 each)

[Boundary Ablation 요약 — Val mDice]
Loss                                                    Val mDice
-----------------------------------------------------------------
CE+Dice                      [baseline]                    0.9324
PLWCE+Dice                   (α=3.158) [baseline]          0.9399
CE+Dice+BL                   [literature]                  0.9399
PLWCE+Dice+BL                (α=3.158)                     0.9423
PLWCE+BL     (no Dice)        (α=3.816)                    0.9333


In [8]:
# === Cell 7: 평가 및 결과 저장 ===

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']
_CMAP  = np.array([[0, 0, 0], [0, 200, 0], [255, 0, 0]], dtype=np.uint8)  # BG=흑 Disc=녹 Cup=적

def colorize_mask(mask_np):
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for c, color in enumerate(_CMAP):
        rgb[mask_np == c] = color
    return rgb

# --- 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
for i, (label, v) in enumerate(all_results.items()):
    c = COLORS[i % len(COLORS)]
    ax1.plot(v['history']['loss'],      label=label[:40], color=c)
    ax2.plot(v['history']['val_mdice'], label=label[:40], color=c)
ax1.set_title('Training Loss');  ax1.set_xlabel('Epoch'); ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)
ax2.set_title('Val mDice');      ax2.set_xlabel('Epoch'); ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_curves.png'), dpi=150)
plt.show()

# --- Test set 정량 평가 ---
print('\n[Test Set 정량 평가]')
print(f"{'Loss':<55} {'mDice':>7} {'Disc':>7} {'Cup':>7}")
print('-' * 78)

final_results = {}
for label, v in all_results.items():
    m = compute_val_metrics(v['model'], test_loader)
    final_results[label] = m
    print(f"{label:<55} {m['mDice']:>7.4f} {m['Dice_Disc']:>7.4f} {m['Dice_Cup']:>7.4f}")

# --- 그룹 바차트 (mDice / Disc / Cup) ---
labels_plot  = list(final_results.keys())
short_labels = [l.split('(')[0].strip() for l in labels_plot]
mdice_v = [final_results[l]['mDice']     for l in labels_plot]
disc_v  = [final_results[l]['Dice_Disc'] for l in labels_plot]
cup_v   = [final_results[l]['Dice_Cup']  for l in labels_plot]

x = np.arange(len(labels_plot))
w = 0.25
fig, ax = plt.subplots(figsize=(15, 6))
ax.bar(x - w, mdice_v, w, label='mDice',     color='steelblue',      alpha=0.85)
ax.bar(x,     disc_v,  w, label='Dice_Disc', color='mediumseagreen', alpha=0.85)
ax.bar(x + w, cup_v,   w, label='Dice_Cup',  color='tomato',         alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(short_labels, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Dice Score'); ax.set_ylim(0, 1.05)
ax.set_title('REFUGE2 Boundary Ablation — Test Dice')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
for xi, (md, di, cu) in zip(x, zip(mdice_v, disc_v, cup_v)):
    ax.text(xi - w, md + 0.003, f'{md:.3f}', ha='center', va='bottom', fontsize=7)
    ax.text(xi,     di + 0.003, f'{di:.3f}', ha='center', va='bottom', fontsize=7)
    ax.text(xi + w, cu + 0.003, f'{cu:.3f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_bar.png'), dpi=150)
plt.show()

# --- 예측 시각화 (best model) ---
best_label = max(final_results, key=lambda k: final_results[k]['mDice'])
best_model = all_results[best_label]['model']
best_model.eval()

sample_imgs, sample_masks = next(iter(test_loader))
with torch.no_grad():
    sample_preds = best_model(sample_imgs.to(device)).argmax(dim=1).cpu()

n_show = min(4, len(sample_imgs))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1: axes = axes[np.newaxis, :]
fig.suptitle(f'REFUGE2 예측 시각화 (Best: {best_label})\nBlack=BG  Green=Disc  Red=Cup')
for i in range(n_show):
    img_vis = (sample_imgs[i].numpy().transpose(1, 2, 0) * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
    axes[i, 0].imshow(img_vis);                             axes[i, 0].set_title('Input'); axes[i, 0].axis('off')
    axes[i, 1].imshow(colorize_mask(sample_masks[i].numpy())); axes[i, 1].set_title('GT');    axes[i, 1].axis('off')
    axes[i, 2].imshow(colorize_mask(sample_preds[i].numpy())); axes[i, 2].set_title('Pred');  axes[i, 2].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boundary_ablation_vis.png'), dpi=100)
plt.show()

# --- JSON 저장 ---
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.json'), 'w', encoding='utf-8') as f:
    json.dump(final_results, f, indent=2, ensure_ascii=False)

# --- Excel 저장 ---
summary_rows = []
for label, m in final_results.items():
    summary_rows.append({
        'Loss_Function': label,
        'mDice':         round(m['mDice'],     4),
        'Dice_Disc':     round(m['Dice_Disc'], 4),
        'Dice_Cup':      round(m['Dice_Cup'],  4),
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss_val, mdice_val) in enumerate(
            zip(v['history']['loss'], v['history']['val_mdice']), 1):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':    ep,
            'Train_Loss': round(loss_val,  6),
            'Val_mDice':  round(mdice_val, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_boundary_ablation.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)

print(f'\n결과 저장 완료: {RESULTS_DIR}')
print('\n=== 최종 결과 요약 ===')
print(df_summary.to_string(index=False))
print(f'\n최고 성능 모델: {best_label}')


[Test Set 정량 평가]
Loss                                                      mDice    Disc     Cup
------------------------------------------------------------------------------
CE+Dice                      [baseline]                  0.9216  0.8447  0.9985
PLWCE+Dice                   (α=3.158) [baseline]        0.9308  0.8627  0.9990
CE+Dice+BL                   [literature]                0.9343  0.8696  0.9990
PLWCE+Dice+BL                (α=3.158)                   0.9315  0.8641  0.9989
PLWCE+BL     (no Dice)        (α=3.816)                  0.9231  0.8475  0.9987

결과 저장 완료: /root/imbalanced-data-LWCE/medical_data/boundary_ablation/results/refuge

=== 최종 결과 요약 ===
                                    Loss_Function  mDice  Dice_Disc  Dice_Cup
          CE+Dice                      [baseline] 0.9216     0.8447    0.9985
PLWCE+Dice                   (α=3.158) [baseline] 0.9308     0.8627    0.9990
        CE+Dice+BL                   [literature] 0.9343     0.8696    0.9990
         